# Create Actual Counts CSV

Run all cells in this notebook to create a single CSV containing the real crowd counts for every image.

In [1]:
from pathlib import Path

import h5py
import pandas as pd

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'DATASET').exists() and (cwd / 'processed_density_maps').exists():
        return cwd
    if (cwd.parent / 'DATASET').exists() and (cwd.parent / 'processed_density_maps').exists():
        return cwd.parent
    raise FileNotFoundError('Could not find project root containing DATASET and processed_density_maps.')


PROJECT_ROOT = resolve_project_root()
DATASET_DIR = PROJECT_ROOT / 'DATASET'
DENSITY_DIR = PROJECT_ROOT / 'processed_density_maps'
ACTUAL_COUNTS_CSV = DATASET_DIR / 'actual_counts.csv'

PROJECT_ROOT

WindowsPath('C:/Users/pendy/Desktop/s6/projects/dip')

In [2]:
def density_sum(h5_path: Path) -> float:
    with h5py.File(h5_path, 'r') as handle:
        density = handle['density'][:]
    return float(density.sum())


rows = []
for part in ['A', 'B']:
    for split in ['train', 'test']:
        image_dir = DATASET_DIR / f'part_{part}' / ('train_data' if split == 'train' else 'test_data') / 'images'
        density_dir = DENSITY_DIR / f'part_{part}' / split
        for image_path in sorted(image_dir.glob('IMG_*.jpg')):
            image_id = image_path.stem
            density_path = density_dir / f'{image_id}.h5'
            if not density_path.exists():
                continue
            rows.append({
                'part': part,
                'split': split,
                'image_id': image_id,
                'image_path': str(image_path.relative_to(PROJECT_ROOT)).replace('\\', '/'),
                'density_path': str(density_path.relative_to(PROJECT_ROOT)).replace('\\', '/'),
                'actual_count': density_sum(density_path),
            })

actual_counts = pd.DataFrame(rows).sort_values(['part', 'split', 'image_id']).reset_index(drop=True)
actual_counts.to_csv(ACTUAL_COUNTS_CSV, index=False)
print(f'Created: {ACTUAL_COUNTS_CSV.relative_to(PROJECT_ROOT).as_posix()}')
actual_counts.head()

Created: DATASET/actual_counts.csv


,part,split,image_id,image_path,density_path,actual_count
0,A,test,IMG_1,DATASET/part_A/test_data/images/IMG_1.jpg,processed_density_maps/part_A/test/IMG_1.h5,172.000000
1,A,test,IMG_10,DATASET/part_A/test_data/images/IMG_10.jpg,processed_density_maps/part_A/test/IMG_10.h5,501.999939
2,A,test,IMG_100,DATASET/part_A/test_data/images/IMG_100.jpg,processed_density_maps/part_A/test/IMG_100.h5,389.000031
3,A,test,IMG_101,DATASET/part_A/test_data/images/IMG_101.jpg,processed_density_maps/part_A/test/IMG_101.h5,210.999969
4,A,test,IMG_102,DATASET/part_A/test_data/images/IMG_102.jpg,processed_density_maps/part_A/test/IMG_102.h5,223.000000


In [3]:
actual_counts.groupby(['split', 'part']).agg(
    images=('image_id', 'count'),
    min_count=('actual_count', 'min'),
    mean_count=('actual_count', 'mean'),
    max_count=('actual_count', 'max')
).round(2)

images  min_count  mean_count  max_count
split part                                          
test  A        182       66.0      432.95     2256.0
      B        316        9.0      123.70      539.0
train A        300       33.0      541.13     3138.0
      B        400       12.0      122.77      576.0